In [8]:
from __future__ import annotations

import re
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd


# ----------------------------
# USER CONFIG
# ----------------------------

MOVIE_NAMES = ["oxygen", "PlaceOfMyBirth"]
TR_DURATION_SEC = 0.8


MOVIE_TIME_RANGES_SEC = {
    "oxygen": [
        (14, 24),
        (228, 230),   # 3:48–3:50
        (380, 385),   # 6:20–6:25
    ],
    "PlaceOfMyBirth": [
        (4*60+24, 4*60+27),
    ],
}

MOVIE_TIMES_CSV = "movie_times.csv"
EVAL_TXT = "evals/eval_median_5_v2_sub-TR.txt"
CASE_INSENSITIVE_MOVIE_MATCH = True


# ----------------------------
# Eval parser
# ----------------------------

@dataclass(frozen=True)
class RunKey:
    sub: str
    ses: str
    run: str


def parse_eval_file(eval_path: str | Path) -> Dict[RunKey, Dict[int, Tuple[float, float]]]:
    """
    Returns:
      { RunKey(sub,ses,run): {TR_index (int): (x,y)} }
    """
    eval_path = Path(eval_path)
    lines = eval_path.read_text(encoding="utf-8", errors="replace").splitlines()

    run_re = re.compile(r"sub-(?P<sub>[^_\/]+)_ses-(?P<ses>[^_\/]+)_run-(?P<run>\d+)")
    row_re = re.compile(r"^\s*(?P<tr>\d+)\s+(?P<x>-?\d+(\.\d+)?)\s+(?P<y>-?\d+(\.\d+)?)\s*$")

    data: Dict[RunKey, Dict[int, Tuple[float, float]]] = {}
    current_key: Optional[RunKey] = None

    for line in lines:
        if line.startswith("RUN:"):
            m = run_re.search(line)
            current_key = RunKey(m.group("sub"), m.group("ses"), m.group("run")) if m else None
            if current_key:
                data.setdefault(current_key, {})
            continue

        if current_key is None:
            continue

        if line.strip().lower().startswith("tr"):
            continue

        m = row_re.match(line)
        if m:
            tr = int(m.group("tr"))
            x = float(m.group("x"))
            y = float(m.group("y"))
            data[current_key][tr] = (x, y)

    return data


# ----------------------------
# Time -> within-movie offsets
# ----------------------------

def tr_offset_floor(seconds: float, tr_dur: float) -> int:
    return int(math.floor(seconds / tr_dur))


def offsets_for_time_range(start_sec: float, end_sec: float, tr_dur: float) -> List[int]:
    """
    Returns all 0-based within-movie TR offsets spanned by [start_sec, end_sec].

    NOTE: This treats end_sec as inclusive-ish (uses floor on end). If you want [start, end)
    (end-exclusive), replace end_off with floor((end_sec - 1e-9)/tr_dur).
    """
    if end_sec < start_sec:
        raise ValueError(f"Bad range: {start_sec}..{end_sec}")

    start_off = tr_offset_floor(start_sec, tr_dur)
    end_off = tr_offset_floor(end_sec, tr_dur)
    return list(range(start_off, end_off + 1))


def norm_movie(s: str) -> str:
    return s.strip().lower()


# ----------------------------
# Main
# ----------------------------

def mean_xy(points: List[Tuple[float, float]]) -> Tuple[float, float]:
    if not points:
        return (float("nan"), float("nan"))
    sx = sum(x for x, _ in points)
    sy = sum(y for _, y in points)
    n = len(points)
    return (sx / n, sy / n)


def main():
    df = pd.read_csv(MOVIE_TIMES_CSV)

    required = {"sub", "ses", "run", "movie", "TR_start", "TR_end"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{MOVIE_TIMES_CSV} missing columns: {sorted(missing)}")

    # Normalize types
    df["sub"] = df["sub"].astype(str)
    df["ses"] = df["ses"].astype(str)
    df["run"] = df["run"].astype(str)
    df["movie"] = df["movie"].astype(str)
    df["TR_start"] = df["TR_start"].astype(int)
    df["TR_end"] = df["TR_end"].astype(int)

    eval_data = parse_eval_file(EVAL_TXT)

    for movie_name in MOVIE_NAMES:
        if CASE_INSENSITIVE_MOVIE_MATCH:
            df_m = df[df["movie"].map(norm_movie) == norm_movie(movie_name)].copy()
        else:
            df_m = df[df["movie"] == movie_name].copy()

        if df_m.empty:
            print(f"[WARN] No rows for movie='{movie_name}' in {MOVIE_TIMES_CSV}")
            continue

        ranges = MOVIE_TIME_RANGES_SEC.get(movie_name, [])
        if not ranges:
            print(f"[WARN] No time ranges specified for movie='{movie_name}'")
            continue
        
        for (start_sec, end_sec) in ranges:
            offsets = offsets_for_time_range(start_sec, end_sec, TR_DURATION_SEC)

            # offset -> list of (x,y) across all occurrences
            buckets: Dict[int, List[Tuple[float, float]]] = {off: [] for off in offsets}

            for _, row in df_m.iterrows():
                key = RunKey(row["sub"], row["ses"], row["run"])
                run_map = eval_data.get(key)
                if run_map is None:
                    continue

                tr_start = int(row["TR_start"])
                tr_end = int(row["TR_end"])

                for off in offsets:
                    abs_tr = tr_start + off
                    if abs_tr < tr_start or abs_tr > tr_end:
                        continue
                    xy = run_map.get(abs_tr)
                    if xy is not None:
                        buckets[off].append(xy)

            # ---- Printing in your requested block format ----
            print(f"{movie_name}, {start_sec}, {end_sec}")
            for off in offsets:
                mx, my = mean_xy(buckets[off])
                print(f"{mx}, {my}")
            print()  # blank line between blocks


if __name__ == "__main__":
    main()


oxygen, 14, 24
-0.8154907741935484, -2.755232612903225
0.4561465483870968, -3.337095290322581
1.3524183225806452, -2.336121580645161
2.860788516129033, -3.0414104193548384
2.7132245161290327, -2.251980354838709
2.7175989677419357, -1.2732003548387096
3.434330483870968, -1.4999285161290323
2.99807635483871, -0.9788624193548389
2.2340023870967745, -0.46790116129032266
2.0317119032258066, 0.008367258064516114
2.410191516129032, -0.8004675161290322
0.7203173548387098, -0.7922972580645162
-1.343458129032258, -0.32894025806451616
-0.942267258064516, -0.26805870967741935

oxygen, 228, 230
1.6434599032258066, -0.81997535483871
0.6689314193548387, -0.9474665806451613
0.29755496774193557, -1.2712407741935483

oxygen, 380, 385
-1.3985590322580648, -0.699168
-2.208136935483871, -0.24015893548387096
-0.735693193548387, -0.8296858064516129
0.2672255806451614, -0.911585806451613
-1.703440387096774, -0.529397516129032
-1.2916691935483873, -0.6100416129032259
-0.4636771612903226, 0.26968122580645154

P